# Class 8: Object oriented programming 2

## Learning outcomes

At the completion of this unit students should be able to:

1. Control attribute access with the `@property` decorator
2. Create and use abstract classes with the `abc` module
3. Apply polymorphism to treat different classes uniformly
4. Handle runtime errors with `try`-`except` blocks and raise exceptions with `raise`
5. Give classes natural behaviour by overriding magic methods like `__str__` and `__add__`

## 8.1 The `property` decorator

Python provides a mechanism for implementing managed attributes. These are class attributes that require some logic when they are being assigned. In other languages, managed attributes are implemented via getter and setter methods.

Let's say a class `A` has a variable `a`. Plain attributes like this one accept **any** value, even nonsense like a negative age:


In [1]:
class A:
    def __init__(self):
        self.a = 0

inst_A = A()
print(inst_A.a)

0


In [2]:
class A:
    def __init__(self):
        self._a = 0
    @property
    def a(self):
        print('Called the getter')
        return self._a

    @a.setter
    def a(self,b):
        print('Called the setter')
        if b < 0:
            self._a = 0
        else:
            self._a = b

inst_A = A()
inst_A.a = -10
print(inst_A.a)


Called the setter
Called the getter
0


In the *managed* version of the class above, `a` is no longer a plain attribute:

- `def a(self)` with the `@property` decorator is the **getter**. Writing `inst_A.a` silently calls it.
- `@a.setter` is the **setter**. Writing `inst_A.a = -10` silently calls it with the value `-10`.
- The data itself is stored in the *private* attribute `self._a` (the leading underscore tells other programmers: hands off, use the property!).

That's why assigning `-10` printed `Called the setter`, and the stored value became `0` instead.

## 8.2 Abstract classes

An abstract class is a class that does not possess (full) implementations for all of its methods. That is, it is a class that is created only to be inherited, and cannot be instantiated; it is "abstract".

Let's take, for example, the class `AbstractClass` below.

In [3]:
class AbstractClass:
    
    def some_method(self):
        pass
    
    
class ChildClass(AbstractClass):
    pass

a = AbstractClass()
c = ChildClass()



This class is not an abstract class because: it can be instantiated (that's variable `a`), and we don't have to implement `some_method()`.

Python doesn't have a native implementation of abstract classes (as is the case in Java, C# and other languages). To implement abstract classes in python, we use the Abstract Base Class module, `abc`, as in the example below. Run this code and check the error you will get.

In [4]:

from abc import ABC, abstractmethod
 
class Animal(ABC):
 
    def __init__(self):
        super().__init__()
        print("Animal is created")
    
    @abstractmethod
    def mobility(self):
        pass

a = Animal()

TypeError: Can't instantiate abstract class Animal with abstract method mobility

Now let's inherit this class.


In [5]:
class Lion(Animal):
  pass

l = Lion()

TypeError: Can't instantiate abstract class Lion with abstract method mobility

The error above happened because we didn't define the method `mobility()`. So we must define it.


In [6]:
class Lion(Animal):
  def mobility(self):
    print('I walk')

l = Lion()
l.mobility()

Animal is created
I walk


Abstract methods are identified by the *decorator* `@abstractmethod` — you can read it as a special marker that Python understands, without needing to know how decorators work internally.

## 8.3 Polymorphism

Polymorphism is Greek for "multiple-forms". It is one of the key OOP concepts, and means that an object variable can have multiple types.

What does this mean? Let's see how this works via an example code.

In [7]:
class Crocodile(): 
     def name(self): 
       print("Crocodile") 
     def mobility(self):
       print("I crawl") 
class Tiger(): 
     def name(self): 
       print("Tiger") 
     def mobility(self):
       print("I run")
      
def call_methods(o): 
    o.name() 
    o.mobility()

o1 = Crocodile() 
o2 = Tiger() 
call_methods(o1) 
call_methods(o2)

Crocodile
I crawl
Tiger
I run


You noticed what happened here? The function `call_methods(o)` calls the methods of some given object variable `o` if those methods exist. If they don't, we get an error.

The interesting bit here is: if `o` is of type `Tiger`, only the `Tiger` methods will be called. If `o` is a `Crocodile`, the `Crocodile` methods will be called.

So `o` inside `call_methods()` can have any type, as long as it has those two methods. Here, we say that `o` is **polymorphic**.

Polymorphism can also work when the classes are related by inheritance. Let's look at this example.


In [8]:

class Animal:
  def intro(self):
    print("This is an animal, it's alive")
  def name(self): 
    print("Animal") 
  def mobility(self):
    print("It depends on the specific animal")

class Crocodile(Animal): 
  def name(self): #Method overriding
    print("Crocodile") 
  def mobility(self):
    print("I crawl") 

class Tiger(Animal): 
  def name(self): 
    print("Tiger") 
  def mobility(self):
    print("I run")
      
def call_methods(o): 
  o.intro()
  o.name() 
  o.mobility()
       
        
o1 = Crocodile() 
o2 = Tiger() 
o3 = Animal()

call_methods(o1) 
call_methods(o2)
call_methods(o3)

This is an animal, it's alive
Crocodile
I crawl
This is an animal, it's alive
Tiger
I run
This is an animal, it's alive
Animal
It depends on the specific animal


Again, `o` is polymorphic inside `call_methods(o)`: it can be of any of the three types. Polymorphism lets python call the method according to the type of the object.

## 8.4 Exception handling

### The `try`-`except` blocks
An exception is an error that happens during the runtime of the program. This is in contrast to a compile-time error, which happens before running the program.

In many OOP languages, exceptions are handled by a special programming construct. In python, if you know that a specific line of code is going to *raise* an exception, you enclose it within a `try-except` block (in Java/C#, it is the `try-catch` block). The syntax is:

```
try:
  problem statements
except ExceptionClass:
  exception handling code
```
Here, when the compiler encounters this statement, it will first attempt to run the `problem statements`. If any of them raises an exception, execution of the statements is terminated, and the `exception handling code` is executed.

For example, what happens when `int()` receives something that isn't a number? `int('abc')` will raise a `ValueError` exception and crash the program:

In [9]:
a = 'abc'
a = int(a)
b = a + 4
print(b)

ValueError: invalid literal for int() with base 10: 'abc'

In [10]:
# simulate a user typing 'abc' first, then a valid number
for typed in ['abc', '8']:
    try:
        a = int(typed)
        b = a + 4
        print(b)
        break
    except:
        print('you should input an integer')

you should input an integer
12


You can also get the information about the raised Exception using the `as` keyword:


```
try:
  problem statements
except ExceptionClass as e:
  exception handling code
```

Here is an example:

And here is `try`/`except` in action, catching the `ValueError` from before and printing its built-in message:

In [11]:
try:
    a = 'abc'
    b = int(a)
except ValueError as e:
    print('You entered a string, try again!')
    print('The error message was:', e)

You entered a string, try again!
The error message was: invalid literal for int() with base 10: 'abc'


### The `try`-`except`-`else`-`finally` blocks

The `try` block in python has 2 blocks, but it can take up to four blocks:

```
try:
  problem statements
except ExceptionClass as e:
  exception handling code
else:
  exception didn't happen
finally:
  execute this block anyway
```

Here is an example:

In [12]:
a=1;b=1
try:
  c=a/b
except:
  print('Woops, something wrong happened!')
else:
  print('No exception happened..')
finally:
  print('Anyway, that was an example of exception handling.')

No exception happened..
Anyway, that was an example of exception handling.


### Explicitly raising an exception: the `raise` keyword

You can raise an exception in your code to interrupt program execution at will. This is by using the `raise` keyword. Here is an example.

In [13]:
print('I am raising an exception now. For no reason.')
raise Exception('An exception was raised without any reason!')

I am raising an exception now. For no reason.


Exception: An exception was raised without any reason!

In [14]:
try:
    a = 1
    raise Exception('oops I did it again!')
except ValueError as ex:
    print(ex)
except Exception as ex:
    print(ex)


oops I did it again!


### Creating custom Exception classes

In the clause `except Exception as e:`, the `Exception` is the name of a class. You've seen several example of various classes that inherit the `Exception` class, such as `ValueError`. You can create your own `Exception` subclass that handles a particular type of errors.

So, the `Exception` class identifies a general exception handler, which handles every exception, whereas exception subclasses handle specific exceptions. You can handle multiple exceptions by adding multiple `except` blocks in your code. The way the exception handler works is top-to-bottom: it searches your `except` blocks, starting from the first one, to an appropriate exception handler. Once it fires the `except` block, no further except blocks will be executed.

Here is an example:

In [15]:
a = 1
b = 0
try:
    c = a / b
    p = [0]
    q = p[1]
except ZeroDivisionError as e:
    print("Woops, division by zero!!")
except Exception as e:
    print("Woops, something wrong happened!")
else:
    print("No exception happened..")
finally:
    print("Anyway, that was an example of exception handling.")

Woops, division by zero!!
Anyway, that was an example of exception handling.


In [16]:
class MyDivByZero(Exception):
  pass

a=1;b=0
try:
  if b==0:
    raise MyDivByZero('MyDivByZero:woops!')
except MyDivByZero:
  print('Woops, MyDivByZero was raised!')
except Exception:
  print('Woops, something wrong happened, but it is generic!')
else:
  print('No exception happened..')
finally:
  print('Anyway, that was an example of exception handling.')

Woops, MyDivByZero was raised!
Anyway, that was an example of exception handling.


Now check out what happens when you put the `except` blocks in the wrong order:

In [17]:
a=1;b=0
try:
  if b==0:
    raise MyDivByZero('MyDivByZero:woops!')
except Exception:
  print('Woops, something wrong happened, but it is generic!')
except MyDivByZero:
  print('Woops, MyDivByZero was raised!')
else:
  print('No exception happened..')
finally:
  print('Anyway, that was an example of exception handling.')

Woops, something wrong happened, but it is generic!
Anyway, that was an example of exception handling.


**GOTO Lab exercises 1-3**

## 8.5 Overriding magic methods in python

Every Python object comes with "magic" methods that can change the behaviour of the object.

- `__init__()`: the constructor, which we already have seen
- `__eq__()`: the equality method, which supports the equality `==` operator
- `__ne__()`: the inequality method, which supports the inequality `!=` operator
- `__add__()`: support the `+` operation on instances of the class
- `__repr__()` and `__str__()`: represents the object as a string
- `__call__()`: implements using the object name as a function call

In [18]:
class Hardisk:
    memory = 1000
    def __init__(self, memory):
        self.memory = memory
        
    def __add__(self, other):
        return Hardisk(self.memory + other.memory)

    def __str__(self):
        return "This hardisk has a memory of " + str(self.memory) + "GB"

    def __eq__(self, other):
        return self.memory == other.memory

    def __call__(self):
        print('You called me')
            
    
h1 = Hardisk(1000)
h2 = Hardisk(1000)
h3 = Hardisk(2000)

h3 = h1 + h2
print(h3)
print(h1)

if h1 == h2:
    print('same')
else:
    print('not the same')

h3()

This hardisk has a memory of 2000GB
This hardisk has a memory of 1000GB
same
You called me


In [19]:
class Nothing:
    def print_stuff(self):
        print('stuff')

n = Nothing()
print(n)
n.print_stuff()

stuff


**GOTO Lab exercises 4-5**

## Summary

- `@property` turns a method into a managed attribute: a getter, a setter, and a private `_attribute` behind it.
- An *abstract class* (`abc.ABC` + `@abstractmethod`) cannot be instantiated until a child class implements its abstract methods.
- *Polymorphism*: the same call `o.mobility()` does different things depending on the actual type of `o` — inheritance is not required, only the method name has to exist.
- `try`/`except` catches exceptions instead of crashing; add `else` (runs on success) and `finally` (runs always); get the error details with `except ValueError as e`.
- `raise` throws an exception deliberately; custom exceptions subclass `Exception`.
- *Magic methods* (`__init__`, `__str__`, `__eq__`, `__add__`, `__call__`, ...) customise how objects behave with `print()`, `==`, `+` and `()`.